# Display JS

> Generates JS functions for updating token display state (caret indicators, highlights, dimming, hidden inputs).

In [ ]:
#| default_exp js.display

In [ ]:
#| export
from cjm_fasthtml_token_selector.core.constants import (
    CARET_INDICATOR_CLS, OPACITY_50_CLS, HIGHLIGHT_CLS,
)
from cjm_fasthtml_token_selector.core.config import TokenSelectorConfig
from cjm_fasthtml_token_selector.core.html_ids import TokenSelectorHtmlIds

In [ ]:
#| export
def _generate_update_inputs_js(
    ids:TokenSelectorHtmlIds,  # HTML IDs
) -> str:  # JS code fragment
    """Generate the hidden input sync function."""
    return f"""
    ns.updateInputs = function() {{
        var anchorEl = document.getElementById('{ids.anchor_input}');
        var focusEl = document.getElementById('{ids.focus_input}');
        if (anchorEl) anchorEl.value = ns.anchor;
        if (focusEl) focusEl.value = ns.focus;
    }};
"""

In [ ]:
#| export
def _generate_gap_display_js(
    ids:TokenSelectorHtmlIds,  # HTML IDs
) -> str:  # JS code fragment
    """Generate gap mode display update function."""
    return f"""
    ns.updateDisplay = function() {{
        var grid = document.getElementById('{ids.token_grid}');
        if (!grid) return;

        // Remove existing caret indicators
        grid.querySelectorAll('.caret-indicator').forEach(function(el) {{ el.remove(); }});

        var tokens = grid.querySelectorAll('.token');
        var prevAnchor = ns._prevAnchor !== undefined ? ns._prevAnchor : -1;

        tokens.forEach(function(token) {{
            var idx = parseInt(token.dataset.tokenIndex);

            // Dimming: tokens before caret
            if (idx < ns.anchor) {{
                if (!token.classList.contains('{OPACITY_50_CLS}')) {{
                    token.classList.add('{OPACITY_50_CLS}');
                }}
            }} else {{
                token.classList.remove('{OPACITY_50_CLS}');
            }}

            // Caret indicator at anchor position
            if (idx === ns.anchor) {{
                var caret = document.createElement('div');
                caret.className = '{CARET_INDICATOR_CLS}';
                token.insertBefore(caret, token.firstChild);
            }}
        }});

        ns._prevAnchor = ns.anchor;
    }};
"""

In [ ]:
#| export
def _generate_word_display_js(
    ids:TokenSelectorHtmlIds,  # HTML IDs
) -> str:  # JS code fragment
    """Generate word mode display update function."""
    return f"""
    ns.updateDisplay = function() {{
        var grid = document.getElementById('{ids.token_grid}');
        if (!grid) return;

        var tokens = grid.querySelectorAll('.token');

        tokens.forEach(function(token) {{
            var idx = parseInt(token.dataset.tokenIndex);

            if (idx === ns.anchor) {{
                if (!token.classList.contains('{HIGHLIGHT_CLS}')) {{
                    token.classList.add('{HIGHLIGHT_CLS}');
                }}
            }} else {{
                token.classList.remove('{HIGHLIGHT_CLS}');
            }}
        }});
    }};
"""

In [ ]:
#| export
def _generate_span_display_js(
    ids:TokenSelectorHtmlIds,  # HTML IDs
) -> str:  # JS code fragment
    """Generate span mode display update function."""
    return f"""
    ns.updateDisplay = function() {{
        var grid = document.getElementById('{ids.token_grid}');
        if (!grid) return;

        // Remove existing caret indicators
        grid.querySelectorAll('.caret-indicator').forEach(function(el) {{ el.remove(); }});

        var tokens = grid.querySelectorAll('.token');
        var lo = Math.min(ns.anchor, ns.focus);
        var hi = Math.max(ns.anchor, ns.focus);

        tokens.forEach(function(token) {{
            var idx = parseInt(token.dataset.tokenIndex);

            // Highlight tokens in range
            if (idx >= lo && idx <= hi) {{
                if (!token.classList.contains('{HIGHLIGHT_CLS}')) {{
                    token.classList.add('{HIGHLIGHT_CLS}');
                }}
            }} else {{
                token.classList.remove('{HIGHLIGHT_CLS}');
            }}

            // Caret at focus position
            if (idx === ns.focus) {{
                var caret = document.createElement('div');
                caret.className = '{CARET_INDICATOR_CLS}';
                token.insertBefore(caret, token.firstChild);
            }}
        }});
    }};
"""

## Public API

In [ ]:
#| export
def generate_display_js(
    config:TokenSelectorConfig,  # config for this instance
    ids:TokenSelectorHtmlIds,    # HTML IDs
) -> str:  # JS code fragment for the IIFE
    """Generate display update and hidden input sync JS functions."""
    display_generators = {
        "gap": _generate_gap_display_js,
        "word": _generate_word_display_js,
        "span": _generate_span_display_js,
    }
    gen = display_generators.get(config.selection_mode, _generate_gap_display_js)
    return _generate_update_inputs_js(ids) + gen(ids)

In [ ]:
from cjm_fasthtml_token_selector.core.config import TokenSelectorConfig, _reset_prefix_counter
from cjm_fasthtml_token_selector.core.html_ids import TokenSelectorHtmlIds

_reset_prefix_counter()

cfg = TokenSelectorConfig(prefix="t", selection_mode="gap")
ids = TokenSelectorHtmlIds(prefix="t")
js = generate_display_js(cfg, ids)
assert "ns.updateDisplay" in js
assert "ns.updateInputs" in js
assert "t-anchor" in js
assert "t-focus" in js
assert "caret-indicator" in js
print("Gap display JS generated!")

cfg_w = TokenSelectorConfig(prefix="tw", selection_mode="word")
ids_w = TokenSelectorHtmlIds(prefix="tw")
js_w = generate_display_js(cfg_w, ids_w)
assert HIGHLIGHT_CLS in js_w
print("Word display JS generated!")

cfg_s = TokenSelectorConfig(prefix="ts", selection_mode="span")
ids_s = TokenSelectorHtmlIds(prefix="ts")
js_s = generate_display_js(cfg_s, ids_s)
assert HIGHLIGHT_CLS in js_s
assert "caret-indicator" in js_s
print("Span display JS generated!")

_reset_prefix_counter()
print("All display JS tests passed!")

Gap display JS generated!
Word display JS generated!
Span display JS generated!
All display JS tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()